# Generate weather data files for different simulation programs with HOSTRADA data

In [1]:
import pvlib
import os
os.environ["HOSTRADA_NETCDF_SUBSET_MODE"] = "full"
#os.environ["HOSTRADA_NETCDF_SUBSET_MODE"] = "auto"
from hostrada4py.hostradaWeatherFileInputs import setup_weather_period_input, setup_location_input
from hostrada4py.hostradaWeatherFileVisualization import setup_weather_file_visualization

In [1]:
from hostrada4py import providerUI as provider_ui

provider_selector = provider_ui.create_provider_selector(
    globals(),
    initial="dwd",
    title="Select weather data source",
    show=True,
)

def _sync_notebook_provider(change=None):
    supported = {value for _, value in provider_ui.variable_options(provider_dropdown.value)}

    current_variable = globals().get("HOSTRADA_VAR")
    if current_variable is not None and current_variable not in supported:
        globals()["HOSTRADA_VAR"] = "tas" if "tas" in supported else next(iter(supported))

    selector = globals().get("variable_selector")
    if selector is not None and hasattr(selector, "options"):
        all_options = globals().get("_hostrada_all_variable_options")
        if all_options is None:
            all_options = list(selector.options)
            globals()["_hostrada_all_variable_options"] = all_options
        filtered = [
            option for option in all_options
            if (option[1] if isinstance(option, tuple) else option) in supported
        ]
        old_values = tuple(value for value in selector.value if value in supported)
        selector.value = ()
        selector.options = filtered
        available = [option[1] if isinstance(option, tuple) else option for option in filtered]
        selector.value = old_values or (("tas",) if "tas" in available else tuple(available[:1]))

provider_dropdown.observe(_sync_notebook_provider, names="value")
_sync_notebook_provider()

## Selecting a location using the OSM map and address search

Select the location for generating weather data by clicking on the map, dragging the marker, entering the longitude and latitude directly, or by entering an address. The variables `selected_lon` and `selected_lat` are automatically updated and used in the subsequent export cells. The center of the map is Berlin-Alexanderplatz.

In [3]:
setup_location_input(globals())

## Time period for the weather file generation

Select the start and end date with the mouse using the date picker widgets. The variables `selected_start` and `selected_end` are automatically updated and used in the subsequent export cells.

In [4]:
setup_weather_period_input(globals())

## Generation of IDA ICE weather Files

In [5]:
from hostrada4py import hostrada_IDA_ICE_Weather as iw
iw.create_ida_ice_weather_file(
    lon=selected_lon, 
    lat=selected_lat, 
    start=selected_start, 
    end=selected_end, 
    output_file="HOSTRADA_IDA_ICE.prn", 
    tz="Europe/Berlin")

Download: https://opendata.dwd.de/climate_environment/CDC/grids_germany/hourly/hostrada/air_temperature_mean/tas_1hr_HOSTRADA-v1-0_BE_gn_2025010100-2025013123.nc
Download: https://opendata.dwd.de/climate_environment/CDC/grids_germany/hourly/hostrada/dew_point/tdew_1hr_HOSTRADA-v1-0_BE_gn_2025010100-2025013123.nc
Download: https://opendata.dwd.de/climate_environment/CDC/grids_germany/hourly/hostrada/humidity_relative/hurs_1hr_HOSTRADA-v1-0_BE_gn_2025010100-2025013123.nc
Download: https://opendata.dwd.de/climate_environment/CDC/grids_germany/hourly/hostrada/pressure_surface/ps_1hr_HOSTRADA-v1-0_BE_gn_2025010100-2025013123.nc
Download: https://opendata.dwd.de/climate_environment/CDC/grids_germany/hourly/hostrada/wind_speed/sfcWind_1hr_HOSTRADA-v1-0_BE_gn_2025010100-2025013123.nc
Download: https://opendata.dwd.de/climate_environment/CDC/grids_germany/hourly/hostrada/wind_direction/sfcWind_direction_1hr_HOSTRADA-v1-0_BE_gn_2025010100-2025013123.nc
Download: https://opendata.dwd.de/climate_e

/Users/nytschgeusen/Desktop/hostrada4py-0.42.0-cerra-provider-dialog-v4/hostrada4py/hostradaDiffuse.py:86: RuntimeWarning: invalid value encountered in divide
  out["kd"] = np.where(ghi.to_numpy() > 0, out["dhi"].to_numpy() / ghi.to_numpy(), 0.0)


PosixPath('HOSTRADA_IDA_ICE.prn')

## Generation of Polysun weather files

In [ ]:
from hostrada4py import hostrada_Polysun_Weather as pw
pw.create_polysun_weather_file(
    lon=selected_lon, 
    lat=selected_lat, 
    start=selected_start, 
    end=selected_end, 
    output_file="HOSTRADA_Polysun.csv", 
    tz="Europe/Berlin")

## Generation of EnergyPlus weather files

In [ ]:
from hostrada4py import hostrada_EnergyPlus_Weather as epw
epw.create_energyplus_weather_file(
    lon=selected_lon, 
    lat=selected_lat, 
    start=selected_start, 
    end=selected_end, 
    output_file="HOSTRADA_EnergyPlus.epw", 
    tz="Europe/Berlin")

## Generation of SimStadt weather files

In [ ]:
from hostrada4py import hostrada_SimStadt_Weather as ssw
ssw.create_simstadt_weather_file(
    lon=selected_lon, 
    lat=selected_lat, 
    start=selected_start, 
    end=selected_end, 
    output_file="HOSTRADA_SimStadt.tmy3", 
    altitude=34.0, tz="Europe/Berlin")

## Generation of BuildingSystems weather files

In [ ]:
from hostrada4py import hostrada_BuildingSystems_Weather as bsw
bsw.create_buildingsystems_csv_weather_file(
    lon=selected_lon, 
    lat=selected_lat, 
    start=selected_start, 
    end=selected_end, 
    output_file="HOSTRADA_BuildingSystems.csv", 
    tz="Europe/Berlin")

## Visualization of the generated weather data

In [ ]:
setup_weather_file_visualization(globals())